# Day 1
# Notebook 1 — Data Gathering and Manipulation (NumPy & Pandas)




## Part 1 — Locating and understanding the data

### 1.1 Repository layout and data location

Start by inspecting the repository to find likely data folders (examples: `data/`, `datasets/`, `raw/`, `structures/`). Use the Jupyter file browser or a short shell listing in a terminal cell.

### 1.2 Loading the main dataset

The workshop uses the tmQM redox dataset distributed with the companion `gnnredox/` clone. Start the kernel from the workspace root so `./gnnredox` resolves, or set `REPO_PATH` to the absolute path of your clone.



In [ ]:
# Standard imports
import os
from pathlib import Path
import pandas as pd

REPO_PATH = os.getcwd() + '/..'

if REPO_PATH is None:
   raise FileNotFoundError(
       'Could not find the gnnredox clone. Clone https://github.com/alvarovm/GraphNetwork-Redox '
       'as gnnredox/ beside this workshop, or set GNNREDOX_PATH.'
   )
print(f'Using gnnredox clone at: {REPO_PATH}')

In [ ]:

# All data containing  complexes and redox potentials
DATA_PATH = REPO_PATH +'/data/tmqm_redox_data_full_data.csv'
df = pd.read_csv(DATA_PATH)
df = df.set_index('csd_code')

print(f'Loaded {len(df)} complexes from: {DATA_PATH}')

# Mol and smiles of complexes generated directly from tmQM dataset
# This also represents the full redox dataset
DATA_PATH = REPO_PATH + '/data/tmc_frm_xyz2mol_tmqm_all_df.pkl'
tmqm_final_df = pd.read_pickle(DATA_PATH)
tmqm_final_df = tmqm_final_df.drop_duplicates(subset='csd_code', keep='first')
tmqm_final_df = tmqm_final_df.set_index('csd_code')

# # Mol and smiles of complexes generated after semi-impirically optimizing the geometries solvated exlicitly with water and then desolvating
# # This also represents the gnn-redox dataset
DATA_PATH = REPO_PATH + '/data/final_df.pkl'
final_df = pd.read_pickle(DATA_PATH)

In [ ]:
tmqm_final_df

### Filters for the gnn-redox dataset

In [ ]:
print(f'Columns in df = {df.columns.tolist()}') 
print(f'    Shape of DF = {df.shape}')
print(f'-'*40)

# final_df.drop(columns=["y"], inplace=True)
print(f'Columns in final_df = {final_df.columns.tolist()}') 
print(f'    Shape of final_df = {final_df.shape}')
print(f'-'*40)
# tmqm_final_df

# tmqm_final_df.drop(columns=["y"], inplace=True)
print(f'Columns in tmqm_final_df = {tmqm_final_df.columns.tolist()}') 
print(f'    Shape of tmqm_final_df = {tmqm_final_df.shape}')


### Merge Dataframes

In [ ]:
df

In [ ]:

df = pd.concat([df, final_df], axis=1, join="inner")


df = pd.concat([df, tmqm_final_df], axis=1, join="inner")

print(f'New Columns in df = {df.columns.tolist()}') 
print(f' New Shape of DF = {df.shape}')


### 1.3 Alternative structural parsing (XYZ-like files)

The structure are represented as ASE atoms objects. The objects are stored in tables in Pickle format, which allows to serialize the python objects.



In [ ]:
# import numpy as np

# df1= pd.read_pickle('/home/vama/soft/hpcbootcamp2026/gnnredox/Data/desolvated_tmqm_all_xyz.pkl')
# df1 = df1.set_index('csd_code')

# df = pd.concat([df, df1], axis=1)

def parse_distances(atomx):
    # atomx = df1.tmqm_atoms[1]
    for k,v in enumerate(atomx):
        if v.symbol == 'Fe':
            fe_atom = k
            break

    distances = atomx.get_all_distances()[fe_atom]
    distances = np.delete(distances,fe_atom)
    return distances



---

## Part 2 — Data inspection with pandas

### 2.1 First look

Show the first few rows and the shape of the data. Use `display()` in Jupyter so wide tables render nicely.



In [ ]:
from IPython.display import display

if 'df' not in globals():
    raise RuntimeError("Load the dataset in the previous cell before running this one.")

print("Shape:", df.shape)
display(df.head())



### 2.2 Data types and summary statistics

Inspect dtypes and a numeric summary. This helps identify columns that need casting or contain unexpected values.



In [ ]:
print(df.dtypes)
display(df.describe(include='all').T)



### 2.3 Column completeness and uniqueness

Check non-null counts and unique values for categorical columns. This gives a quick picture of missingness and label cardinality.



In [ ]:
nulls = df.isnull().sum().sort_values(ascending=False)
print(nulls[nulls>0])

cat_cols = df.select_dtypes(include=['object','category' ]).columns.tolist()

cat_cols.remove('desolv_atoms')
cat_cols.remove('tmqm_atoms')
cat_cols.remove('atoms')
cat_cols
for c in cat_cols:
    print(c, "-> unique:", df[c].nunique())



### 2.4 Physical meaning of tmQM columns

| Column | Meaning | Notes |
|---|---:|---|
| `csd_code` | Cambridge Structural Database identifier | unique complex identifier |
| `q` | Overall complex charge | integer |
| `Stoichiometry` | Elemental composition | compact formula, e.g. `C12H14FeN4O10` |
| `num_atoms` | Number of atoms | integer |
| `ligands_list` | Serialized ligand SMILES list | supports ligand-count features |
| `tm_oxs` | Transition-metal oxidation state | constant for this Fe(II) subset |
| `ligands_q_list` | Serialized ligand formal charges | supports charge-derived features |
| `reduction_pot` | Experimental reduction potential | regression target in volts |

### 2.5 Feature engineering from composition and ligands

The raw table has only two useful native numeric predictors (`q` and `num_atoms`); `tm_oxs` is constant. Derive composition and ligand features so PCA and the baseline models have meaningful inputs.



In [ ]:
import ast
import re

TARGET_COL = 'reduction_pot'
if TARGET_COL not in df.columns:
    raise RuntimeError(f"Expected target column '{TARGET_COL}' was not found")

def parse_stoichiometry(formula):
    return {
        element: int(count) if count else 1
        for element, count in re.findall(r'([A-Z][a-z]?)(\d*)', formula)
    }

stoichiometry = df['Stoichiometry'].map(parse_stoichiometry)
for element in ['C', 'H', 'N', 'O', 'S', 'Cl', 'P']:
    df[f'n_{element}'] = stoichiometry.map(lambda counts, el=element: counts.get(el, 0))

df['n_ligands'] = df['ligands_list'].map(lambda value: len(ast.literal_eval(value)))
df['q_sum'] = df['ligands_q_list'].map(lambda value: sum(ast.literal_eval(value)))
df['n_anionic'] = df['ligands_q_list'].map(
    lambda value: sum(charge < 0 for charge in ast.literal_eval(value))
)

# constant_cols = [c for c in df.select_dtypes(include='number') if df[c].nunique() <= 1]
# if constant_cols:
#     print('Dropping constant columns:', constant_cols)
#     df = df.drop(columns=constant_cols)

numeric_cols = df.select_dtypes(include='number').columns.tolist()
FEATURE_COLS = [c for c in numeric_cols if c != TARGET_COL]
print('Target:', TARGET_COL)
print('Numeric features:', FEATURE_COLS)



### 2.6 Feature engineering: compute Fe–ligand bond lengths

We compute Euclidean distances between Fe coordinates and ligand atom coordinates. In 3D, the Euclidean distance is:

$$
d_{ij} = \lVert \mathbf{r}_i - \mathbf{r}_j \rVert_2 = \sqrt{\sum_{k=1}^{3} (r_{i,k} - r_{j,k})^2}
$$

Implement a helper that extracts Fe coordinates from a row and computes bond-length statistics (min, max, mean) to be used as features.



In [ ]:
import math


def compute_fe_bond_lengths(row):
    atomx = row.desolv_atoms
    dists = parse_distances(atomx)
    return pd.Series({'fe_bond_min':  float(np.nanmin(dists)),
                      'fe_bond_mean': float(np.nanmean(dists)),
                      'fe_bond_max':  float(np.nanmax(dists))})


# Apply to the dataframe (may be slow; consider vectorized approaches for large datasets)
# if 'fe_x' in df.columns:
fe_features = df.apply(compute_fe_bond_lengths, axis=1)
df = pd.concat([df, fe_features], axis=1)
print('Computed Fe bond-length features')
# else:
#     print('fe_x/fe_y/fe_z not found; skipping Fe bond-length computation')



---

## Part 3 — Data cleaning

### 3.1 Handling missing values

Strategy: prefer dropping rows with missing target values; for features, either impute (mean/median) or flag missingness via an indicator column.



In [ ]:
# Drop rows missing the target
before = len(df)
df = df[df[TARGET_COL].notnull()].copy()
print(f"Dropped {before - len(df)} rows with missing target '{TARGET_COL}'")

# Impute numeric features with median to be robust to outliers
num_cols = [c for c in df.select_dtypes(include='number') if c != TARGET_COL]
for c in num_cols:
    if df[c].isnull().any():
        med = df[c].median()
        df[f"{c}_was_missing"] = df[c].isnull()
        df[c] = df[c].fillna(med)
        print(f"Imputed {c} with median = {med}")



**Mentor checkpoint 3**

- Confirm dataset selection and target column are correct for the group.
- Discuss any domain-specific filters (oxidation state, solvent, basis set) to apply before modeling.
- Decide whether to drop outliers flagged by `is_outlier_any` or to keep them and add robust models later.

Proceed only after confirmation.

---

### Exercise 1.1 — Quick data exploration (10 minutes)

1. Print the distribution (value counts) of a categorical column such as `ligand_type` or `solvent` if present.
2. Report the number of unique Fe centers (hint: group by a unique complex identifier).
3. Compute the fraction of rows that were flagged as outliers.

Write code to answer these questions and print short conclusions (1-2 lines each).

### Exercise 1.2 — NumPy operations on structure arrays (20 minutes)

1. Using the `parse_xyz_file` helper, read one structure file and compute the centroid of its atoms. Use NumPy vector operations.

Recall z-score normalization for a sample:

$$
z_i = \frac{x_i - \mu}{\sigma}
$$

and the sample covariance between X and Y:

$$
\mathrm{Cov}(X,Y) = \frac{1}{n-1}\sum_{i=1}^{n} (x_i - \bar{x})(y_i - \bar{y})
$$

2. Compute the z-scores for `n_O` and compute its covariance with `reduction_pot`.

### Exercise 1.3 — Save the cleaned dataset (5 minutes)

Save `df` to `data_cleaned.csv` in the notebook folder. This file will be used by Notebook 2.



In [ ]:
# OUTPUT_DIR = Path(os.environ.get('SCRATCH', '.')) / 'Fe-Redox-GNN'
thispath = os.getcwd()
OUTPUT_DIR =  thispath + '/../output'
if not os.path.isdir(OUTPUT_DIR):
    os.mkdir(OUTPUT_DIR)

CLEANED_PATH = OUTPUT_DIR +'/data_cleaned.pkl'
df.to_pickle(CLEANED_PATH)#, index=False)
print('Wrote', CLEANED_PATH, 'with', len(df), 'rows')



---

## Summary

| Task | Tool | Key functions |
|---|---:|---|
| Load data | pandas | `pd.read_csv` |
| Inspect | pandas | `df.head`, `df.describe`, `df.dtypes` |
| Outlier detection | pandas, numpy | quantiles, boolean masks |
| Feature engineering | numpy, custom | `compute_fe_bond_lengths` |

---



## Preparation for Day 2
Before tomorrow:
- Ensure your `fe-redox` conda environment is activated and working
- Explore the cloned repo and identify the data files (CSV/NPY/PKL) used by the project
- Brush up on Pandas basics (Pandas 10-minute tutorial: https://pandas.pydata.org/docs/getting_started/10min.html)

We will load and clean the dataset, visualize structure-property relationships, and prepare features for ML models on Day 2. We will cover dimensionality reduction (PCA), clustering (K-Means), and classical ML baselines (Random Forest, Gaussian Process Regression). Make sure `data_cleaned.csv` exists and confirm the shortlist of features you want to try as baselines.